# EU CELLAR Climate Law ETL Pipeline

**Project:** Empirical research on hybrid retrieval (lexical + semantic) for legal RAG systems  
**Goal:** Extract article-level structured data from primary EU climate law documents  
**Pipeline stages:**
1. SPARQL batch → CELEX IDs + Cellar work URIs
2. SPARQL per-doc → fmx4 (Formex) or html manifestation URLs
3. Cellar REST → Formex zip downloads (fmx4) or HTML fallback
4. BeautifulSoup → Article-level records
5. pandas → `.jsonl` export

**EuroVoc concept cluster (climate + energy):**
| URI | Label |
|-----|-------|
| eurovoc/5482 | Climate change |
| eurovoc/434743 | Climate change policy |
| eurovoc/5641 | Greenhouse gas |
| eurovoc/434747 | Reduction of gas emissions |
| eurovoc/434745 | Emission trading |
| eurovoc/6645 | Renewable energy |
| eurovoc/6642 | Energy efficiency |

**Cellar API notes:**
- CELEX sector-3 Directives use type-letter `L`; Regulations use `R`
- The public SPARQL endpoint only exposes expression/manifestation triples when the work URI is hardcoded in the query — a known graph-boundary limitation
- fmx4 manifestations are delivered as zips (`Accept: application/zip`); articles are `<ARTICLE>` elements inside an `<ACT>` root
- HTML manifestations (`publications.europa.eu`, not `eur-lex.europa.eu`) are accessible without WAF and used as fallback

## 0. Imports & Configuration

In [1]:
import copy
import io
import json
import re
import time
import logging
import zipfile
from pathlib import Path
from typing import Optional

import pandas as pd
import requests
from bs4 import BeautifulSoup
from SPARQLWrapper import SPARQLWrapper, JSON as SPARQL_JSON
from tqdm.auto import tqdm

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

CELLAR_SPARQL   = "http://publications.europa.eu/webapi/rdf/sparql"
LANG_ENG_URI    = "http://publications.europa.eu/resource/authority/language/ENG"

RAW_CACHE_DIR   = Path("../data/etl/raw_cache")   # downloaded content (zips + html)
MANIFEST_CACHE  = Path("../data/etl/manifest_cache.json")  # fmx4/html URLs per CELEX
OUTPUT_JSONL    = Path("../data/corpus/eu_climate_articles.jsonl")

RAW_CACHE_DIR.mkdir(exist_ok=True)

print("Environment ready.")
print(f"  Cache dir    : {RAW_CACHE_DIR.resolve()}")
print(f"  Output JSONL : {OUTPUT_JSONL.resolve()}")

Environment ready.
  Cache dir    : /Users/bitbase/workspace/github.com/rameezshafat/eu-lexical-semantic-legal-rag/raw_cache
  Output JSONL : /Users/bitbase/workspace/github.com/rameezshafat/eu-lexical-semantic-legal-rag/eu_climate_articles.jsonl


/Users/bitbase/workspace/github.com/rameezshafat/eu-lexical-semantic-legal-rag/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## Phase 1a — Batch SPARQL: CELEX IDs + Cellar Work URIs

Retrieve every sector-3 Directive (`L`) and Regulation (`R`) tagged with the seven EuroVoc climate/energy concepts.  
Also fetches the Cellar UUID (work URI) needed for the per-document manifestation lookup.

In [2]:
EUROVOC_CONCEPTS = [
    "<http://eurovoc.europa.eu/5482>",
    "<http://eurovoc.europa.eu/434743>",
    "<http://eurovoc.europa.eu/5641>",
    "<http://eurovoc.europa.eu/434747>",
    "<http://eurovoc.europa.eu/434745>",
    "<http://eurovoc.europa.eu/6645>",
    "<http://eurovoc.europa.eu/6642>",
]

VALUES_BLOCK = " ".join(EUROVOC_CONCEPTS)

SPARQL_BATCH = f"""
PREFIX cdm: <http://publications.europa.eu/ontology/cdm#>

SELECT DISTINCT ?celex_id ?work
WHERE {{
  VALUES ?concept {{ {VALUES_BLOCK} }}

  ?work cdm:work_is_about_concept_eurovoc ?concept ;
        cdm:resource_legal_id_celex       ?celex_id .

  # Sector 3, type R (Regulation) or L (Directive)
  FILTER (REGEX(STR(?celex_id), "^3[0-9]{{4}}[RL]"))
}}
ORDER BY ?celex_id
"""

def run_sparql(endpoint: str, query: str, timeout: int = 120) -> list[dict]:
    sp = SPARQLWrapper(endpoint)
    sp.setQuery(query)
    sp.setReturnFormat(SPARQL_JSON)
    sp.setTimeout(timeout)
    r = sp.query().convert()
    return [{var: b[var]["value"] for var in b} for b in r["results"]["bindings"]]


log.info("Running batch SPARQL for CELEX IDs + work URIs …")
batch_rows = run_sparql(CELLAR_SPARQL, SPARQL_BATCH)

# Deduplicate by CELEX ID (same work may appear under multiple concepts)
work_map: dict[str, str] = {}  # celex_id → cellar work URI
for row in batch_rows:
    cid  = row["celex_id"]
    work = row["work"]
    if cid not in work_map:
        work_map[cid] = work

celex_ids = list(work_map.keys())

def doc_type(cid: str) -> str:
    return "Directive" if cid[5] == "L" else "Regulation"

dirs  = [c for c in celex_ids if doc_type(c) == "Directive"]
regs  = [c for c in celex_ids if doc_type(c) == "Regulation"]

print(f"\n✓ SPARQL returned {len(batch_rows)} row(s), {len(celex_ids)} unique CELEX IDs.")
print(f"  Directives  : {len(dirs)}")
print(f"  Regulations : {len(regs)}")
print()
print("All CELEX IDs:")
for cid in celex_ids:
    print(f"  [{doc_type(cid):10s}] {cid}")

20:17:57 [INFO] Running batch SPARQL for CELEX IDs + work URIs …



✓ SPARQL returned 131 row(s), 131 unique CELEX IDs.
  Directives  : 23
  Regulations : 108

All CELEX IDs:
  [Directive ] 32003L0030
  [Directive ] 32003L0087
  [Directive ] 32003L0087R(02)
  [Directive ] 32003L0087R(03)
  [Directive ] 32003L0087R(04)
  [Directive ] 32003L0087R(05)
  [Directive ] 32003L0087R(06)
  [Directive ] 32008L0101
  [Directive ] 32008L0101R(02)
  [Directive ] 32008L0101R(03)
  [Directive ] 32008L0101R(04)
  [Regulation] 32013R0525
  [Regulation] 32013R0525R(01)
  [Regulation] 32013R0525R(02)
  [Regulation] 32013R1293
  [Regulation] 32013R1293R(01)
  [Regulation] 32014R0215
  [Regulation] 32014R0215R(01)
  [Regulation] 32014R0215R(02)
  [Regulation] 32014R0517
  [Regulation] 32014R0517R(01)
  [Regulation] 32014R0517R(02)
  [Regulation] 32014R0517R(03)
  [Regulation] 32014R0517R(04)
  [Regulation] 32014R0517R(05)
  [Regulation] 32014R0517R(06)
  [Regulation] 32014R0517R(07)
  [Regulation] 32014R0517R(08)
  [Regulation] 32014R0517R(09)
  [Regulation] 32014R0517R(1

---
## Phase 1b — Per-Document SPARQL: Manifestation URL Lookup

The CELLAR SPARQL endpoint only exposes expression/manifestation triples when the **work URI is hardcoded** in the query pattern (a known graph-boundary limitation of the public endpoint).  
We therefore run one lightweight SPARQL query per document using the cellar UUID we retrieved in Phase 1a.

Priority order: **fmx4** (Formex XML zip, structured article tags) → **html** (plain HTML, article-text regex split) → skip.

In [3]:
SPARQL_DELAY = 0.4   # seconds between per-doc SPARQL calls


def get_manifestations(work_uri: str) -> dict[str, str]:
    """
    Return {format_type: manifestation_url} for the English expression of *work_uri*.
    Requires the work URI to be hardcoded; variable bindings do not traverse this graph.
    """
    Q = f"""
PREFIX cdm: <http://publications.europa.eu/ontology/cdm#>
SELECT ?manif ?mtype
WHERE {{
  ?expr cdm:expression_belongs_to_work <{work_uri}> ;
        cdm:expression_uses_language   <{LANG_ENG_URI}> .
  ?manif cdm:manifestation_manifests_expression ?expr .
  OPTIONAL {{ ?manif cdm:manifestation_type ?mtype }}
}}
"""
    rows = run_sparql(CELLAR_SPARQL, Q, timeout=15)
    return {r.get("mtype", "unknown"): r["manif"] for r in rows}


def best_manifest_url(manifests: dict[str, str]) -> tuple[str, str]:
    """Return (format_type, url) for the best available format."""
    for fmt in ("fmx4", "xhtml", "html"):
        if fmt in manifests:
            return fmt, manifests[fmt]
    return "", ""


# Load cached manifest URLs if available (speeds up re-runs)
if MANIFEST_CACHE.exists():
    manifest_store: dict[str, dict] = json.loads(MANIFEST_CACHE.read_text())
    print(f"Loaded manifest cache: {len(manifest_store)} entries.")
else:
    manifest_store = {}

print(f"Will look up manifestations for {len(celex_ids)} document(s) …")

Will look up manifestations for 131 document(s) …


In [4]:
skipped: list[str] = []

for celex_id in tqdm(celex_ids, desc="Manifest lookup"):
    if celex_id in manifest_store:
        continue  # already cached

    work_uri   = work_map[celex_id]
    manifests  = get_manifestations(work_uri)
    fmt, url   = best_manifest_url(manifests)

    manifest_store[celex_id] = {"fmt": fmt, "url": url, "work_uri": work_uri}

    if not url:
        log.warning("  No usable manifest for %s  (available: %s)", celex_id, list(manifests.keys()))
        skipped.append(celex_id)

    time.sleep(SPARQL_DELAY)

# Persist the manifest cache
MANIFEST_CACHE.write_text(json.dumps(manifest_store, indent=2))

fmx4_docs  = [c for c, m in manifest_store.items() if m["fmt"] == "fmx4"]
html_docs  = [c for c, m in manifest_store.items() if m["fmt"] in ("html", "xhtml")]
no_docs    = [c for c, m in manifest_store.items() if not m["url"]]

print(f"\n✓ Manifest lookup complete.")
print(f"  fmx4 (Formex zip) : {len(fmx4_docs)}")
print(f"  html / xhtml      : {len(html_docs)}")
print(f"  No manifest found : {len(no_docs)}")

Manifest lookup:   0%|          | 0/131 [00:00<?, ?it/s]

Manifest lookup:   1%|          | 1/131 [00:00<01:33,  1.39it/s]

Manifest lookup:   2%|▏         | 2/131 [00:01<01:33,  1.37it/s]

20:18:00 [WARNING]   No usable manifest for 32003L0087R(02)  (available: [])


Manifest lookup:   2%|▏         | 3/131 [00:02<01:34,  1.36it/s]

...

Manifest lookup:  98%|█████████▊| 129/131 [01:37<00:01,  1.27it/s]

Manifest lookup:  99%|█████████▉| 130/131 [01:38<00:00,  1.31it/s]

Manifest lookup: 100%|██████████| 131/131 [01:39<00:00,  1.32it/s]

Manifest lookup: 100%|██████████| 131/131 [01:39<00:00,  1.32it/s]


✓ Manifest lookup complete.
  fmx4 (Formex zip) : 58
  html / xhtml      : 1
  No manifest found : 72


---
## Phase 2 — Fetch Content

- **fmx4**: `Accept: application/zip` → zip archive containing Formex XML
- **html/xhtml**: `Accept: text/html` → plain HTML served directly from `publications.europa.eu`

Both types are cached locally.

In [5]:
INTER_REQUEST_DELAY = 1.2
RETRY_DELAYS        = [5, 15, 30]

ACCEPT_MAP = {
    "fmx4" : "application/zip",
    "xhtml": "application/xhtml+xml, text/html;q=0.9",
    "html" : "text/html, application/xhtml+xml;q=0.9",
}


def fetch_content(celex_id: str, fmt: str, url: str) -> Optional[bytes]:
    """
    Download content for *celex_id*. Returns raw bytes or None on failure.
    Results cached under RAW_CACHE_DIR as {celex_safe}.{fmt}.
    """
    if not url:
        return None

    safe_name  = re.sub(r"[^A-Za-z0-9_-]", "_", celex_id)
    ext        = "zip" if fmt == "fmx4" else "html"
    cache_path = RAW_CACHE_DIR / f"{safe_name}.{ext}"

    if cache_path.exists():
        log.debug("[CACHE] %s", celex_id)
        return cache_path.read_bytes()

    headers = {
        "Accept"    : ACCEPT_MAP.get(fmt, "*/*"),
        "User-Agent": "eu-climate-rag-research/1.0 (academic)",
    }

    for attempt, backoff in enumerate([0] + RETRY_DELAYS, start=1):
        if backoff:
            log.warning("  Retry %d for %s — waiting %ds", attempt, celex_id, backoff)
            time.sleep(backoff)
        try:
            resp = requests.get(url, headers=headers, timeout=60, allow_redirects=True)
            ct   = resp.headers.get("Content-Type", "")

            ok = (
                resp.status_code == 200
                and len(resp.content) > 100
                and (
                    (fmt == "fmx4" and "zip" in ct)
                    or (fmt in ("html", "xhtml") and ("html" in ct or "xml" in ct))
                )
            )
            if ok:
                cache_path.write_bytes(resp.content)
                log.info("[OK] %s (%s, %d KB)", celex_id, fmt, len(resp.content) // 1024)
                time.sleep(INTER_REQUEST_DELAY)
                return resp.content

            if resp.status_code in (429, 503):
                continue

            log.error("[%d] %s — skipping", resp.status_code, celex_id)
            return None

        except requests.RequestException as exc:
            log.warning("  Network error for %s: %s", celex_id, exc)

    log.error("All retries exhausted for %s", celex_id)
    return None


print("Fetch function defined.")

Fetch function defined.


In [6]:
content_store: dict[str, tuple[str, bytes]] = {}  # celex_id → (fmt, bytes)
fetch_failed:  list[str] = []

to_fetch = [(cid, m["fmt"], m["url"]) for cid, m in manifest_store.items() if m["url"]]
print(f"Fetching {len(to_fetch)} document(s) …\n")

for celex_id, fmt, url in tqdm(to_fetch, desc="Downloading"):
    data = fetch_content(celex_id, fmt, url)
    if data:
        content_store[celex_id] = (fmt, data)
    else:
        fetch_failed.append(celex_id)

print(f"\n✓ Downloaded : {len(content_store)}")
print(f"  Failed     : {len(fetch_failed)}")
if fetch_failed:
    print("  Failed IDs :", fetch_failed)

Fetching 59 document(s) …



Downloading:   0%|          | 0/59 [00:00<?, ?it/s]

20:19:38 [INFO] [OK] 32003L0030 (html, 22 KB)


Downloading:   2%|▏         | 1/59 [00:01<01:51,  1.93s/it]

20:19:40 [INFO] [OK] 32003L0087 (fmx4, 23 KB)


...

Downloading:  98%|█████████▊| 58/59 [01:58<00:01,  1.89s/it]

20:21:36 [INFO] [OK] 32026R0893 (fmx4, 5 KB)


Downloading: 100%|██████████| 59/59 [02:00<00:00,  1.85s/it]

Downloading: 100%|██████████| 59/59 [02:00<00:00,  2.04s/it]


✓ Downloaded : 59
  Failed     : 0


---
## Phase 3 — Parsing (Article-Level Extraction)

### Formex XML (fmx4)
The zip contains several XML files; the main act has root `<ACT>` with `<ARTICLE>` elements:  
- `<TI.ART>` — article title
- `<STI.ART>` — optional sub-title
- `<PARAG>` → `<ALINEA>` — paragraph text
- `<ALINEA>` directly under `<ARTICLE>` — single-paragraph articles
- `<REF.DOC>` — cross-reference nodes

### HTML fallback
The HTML is flat (no structured article tags). Articles are split by matching `Article N` header patterns in the plain text.

In [7]:
# ── Formex helpers ─────────────────────────────────────────────────────────────

def _leaf_alineas(art_node) -> list:
    """Return <ALINEA> nodes not nested inside another <ALINEA>."""
    all_alineas = art_node.find_all("ALINEA")
    leaf = []
    for a in all_alineas:
        parent, is_nested = a.parent, False
        while parent and parent != art_node:
            if parent.name == "ALINEA":
                is_nested = True
                break
            parent = parent.parent
        if not is_nested:
            leaf.append(a)
    return leaf


def fmx_article_number(art_node) -> str:
    ti = art_node.find("TI.ART")
    if ti:
        return ti.get_text(separator=" ", strip=True)
    no = art_node.get("IDENTIFIER") or art_node.get("NO") or ""
    return str(no).strip()


def fmx_article_text(art_node) -> str:
    parts = []
    sti = art_node.find("STI.ART")
    if sti:
        parts.append(sti.get_text(separator=" ", strip=True))

    alineas = _leaf_alineas(art_node)
    if alineas:
        parts += [a.get_text(separator=" ", strip=True) for a in alineas]
    else:
        p_tags = art_node.find_all("P")
        if p_tags:
            parts += [p.get_text(separator=" ", strip=True) for p in p_tags]
        else:
            ac = copy.copy(art_node)
            ti = ac.find("TI.ART")
            if ti:
                ti.decompose()
            parts.append(ac.get_text(separator=" ", strip=True))

    return "\n".join(p for p in parts if p)


def fmx_cross_refs(art_node) -> list[str]:
    refs = []
    for ref in art_node.find_all("REF.DOC"):
        val = ref.get("CELEX") or ref.get("FILE") or ref.get_text(strip=True)
        if val:
            refs.append(val)
    return refs


def parse_fmx4_zip(celex_id: str, doc_type: str, zip_bytes: bytes) -> list[dict]:
    """Parse all XML files in a Formex zip; return one record per article."""
    z = zipfile.ZipFile(io.BytesIO(zip_bytes))
    records = []
    for fname in z.namelist():
        if not fname.endswith(".xml") or fname.endswith((".toc.xml", ".doc.xml")):
            continue
        try:
            xml_text = z.read(fname).decode("utf-8", errors="replace")
            soup     = BeautifulSoup(xml_text, "xml")
            for art in soup.find_all(["ARTICLE", "ART"]):
                records.append({
                    "celex_id"        : celex_id,
                    "doc_type"        : doc_type,
                    "article_number"  : fmx_article_number(art),
                    "article_text"    : fmx_article_text(art),
                    "cross_references": fmx_cross_refs(art),
                })
        except Exception as exc:
            log.warning("  Could not parse %s in %s: %s", fname, celex_id, exc)
    return records


# ── HTML fallback helper ────────────────────────────────────────────────────────

# Matches "Article 1", "Article 2a", "ARTICLE 1", etc.
_ART_RE = re.compile(r"(?:^|\n)(Article\s+(\d+[a-z]?)(?:\s*[–\-]\s*(.+?))?(?=\n|$))", re.IGNORECASE | re.MULTILINE)


def parse_html(celex_id: str, doc_type: str, html_bytes: bytes) -> list[dict]:
    """Extract articles from a flat HTML document by splitting on 'Article N' headers."""
    soup = BeautifulSoup(html_bytes.decode("utf-8", errors="replace"), "html.parser")
    text = soup.get_text(separator="\n")

    # Find all article header positions
    matches = list(_ART_RE.finditer(text))
    if not matches:
        return []

    records = []
    for i, m in enumerate(matches):
        start     = m.start()
        end       = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        art_num   = m.group(0).strip()
        art_text  = text[start:end].strip()

        records.append({
            "celex_id"        : celex_id,
            "doc_type"        : doc_type,
            "article_number"  : art_num,
            "article_text"    : art_text,
            "cross_references": [],
        })
    return records


print("Parser functions defined.")

Parser functions defined.


In [8]:
all_records:  list[dict] = []
parse_errors: list[str]  = []

print(f"Parsing {len(content_store)} document(s) …\n")

for celex_id, (fmt, data) in tqdm(content_store.items(), desc="Parsing"):
    dtype = doc_type(celex_id)
    try:
        if fmt == "fmx4":
            records = parse_fmx4_zip(celex_id, dtype, data)
        else:
            records = parse_html(celex_id, dtype, data)

        all_records.extend(records)
        log.info("  %s (%s) → %d article(s)", celex_id, fmt, len(records))
    except Exception as exc:
        log.error("  Parse failed for %s: %s", celex_id, exc)
        parse_errors.append(celex_id)

print(f"\n✓ Total article records : {len(all_records)}")
print(f"  Parse errors          : {len(parse_errors)}")
if parse_errors:
    print("  Error IDs:", parse_errors)

Parsing 59 document(s) …



Parsing:   0%|          | 0/59 [00:00<?, ?it/s]

20:21:38 [INFO]   32003L0030 (html) → 9 article(s)


20:21:38 [INFO]   32003L0087 (fmx4) → 33 article(s)


20:21:38 [INFO]   32008L0101 (fmx4) → 16 article(s)


...

20:21:39 [INFO]   32026R0285 (fmx4) → 5 article(s)


20:21:39 [INFO]   32026R0667 (fmx4) → 3 article(s)


20:21:39 [INFO]   32026R0893 (fmx4) → 2 article(s)


Parsing: 100%|██████████| 59/59 [00:01<00:00, 41.86it/s]


✓ Total article records : 825
  Parse errors          : 0


In [9]:
if all_records:
    sample = all_records[0]
    print("── Sample article record ──────────────────────────────────────")
    for key, val in sample.items():
        display_val = str(val)[:250] + "…" if len(str(val)) > 250 else val
        print(f"  {key:20s}: {display_val}")

── Sample article record ──────────────────────────────────────
  celex_id            : 32003L0030
  doc_type            : Directive
  article_number      : Article 1
  article_text        : Article 1
This Directive aims at promoting the use of biofuels or other renewable fuels to replace diesel or petrol for transport purposes in each Member State, with a view to contributing to objectives such as meeting climate change commitments, env…
  cross_references    : []


---
## Phase 4 — Data Structuring & Export

In [10]:
COLUMNS = ["celex_id", "doc_type", "article_number", "article_text", "cross_references"]

df = pd.DataFrame(all_records, columns=COLUMNS) if all_records else pd.DataFrame(columns=COLUMNS)

print(f"Raw DataFrame shape : {df.shape}")
df.info()
if not df.empty:
    print(df.head(3).to_string())

Raw DataFrame shape : (825, 5)
<class 'pandas.DataFrame'>
RangeIndex: 825 entries, 0 to 824
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   celex_id          825 non-null    str   
 1   doc_type          825 non-null    str   
 2   article_number    825 non-null    str   
 3   article_text      825 non-null    str   
 4   cross_references  825 non-null    object
dtypes: object(1), str(4)
memory usage: 32.4+ KB
     celex_id   doc_type article_number                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [11]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[\s]+", " ", text)
    text = re.sub(r"[\x00-\x08\x0b-\x1f\x7f]", "", text)
    return text.strip()


if not df.empty:
    df["article_number"] = df["article_number"].apply(clean_text)
    df["article_text"]   = df["article_text"].apply(clean_text)

    before = len(df)
    df = df[df["article_text"].str.len() > 0].reset_index(drop=True)
    print(f"Rows before cleaning : {before}")
    print(f"Rows after  cleaning : {len(df)}  (dropped {before - len(df)} empty)")
else:
    print("DataFrame is empty.")

Rows before cleaning : 825
Rows after  cleaning : 825  (dropped 0 empty)


In [12]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

if df.empty:
    print("No records.")
else:
    print(f"Total articles         : {len(df):,}")
    print(f"Unique documents       : {df['celex_id'].nunique():,}")
    print()
    print("Articles by doc_type:")
    print(df.groupby("doc_type").size().to_string())
    print()
    print("Articles per CELEX ID (top 20):")
    print(df.groupby(["celex_id","doc_type"]).size().head(20).to_string())
    print()
    print("Article text length stats (chars):")
    print(df["article_text"].str.len().describe().to_string())
    print()
    cross_ref_counts = df["cross_references"].apply(len)
    print(f"Total cross-references : {cross_ref_counts.sum():,}")
    print(f"Avg refs per article   : {cross_ref_counts.mean():.2f}")

DATASET SUMMARY
Total articles         : 825
Unique documents       : 57

Articles by doc_type:
doc_type
Directive     122
Regulation    703

Articles per CELEX ID (top 20):
celex_id    doc_type  
32003L0030  Directive      9
32003L0087  Directive     33
32008L0101  Directive     16
32013R0525  Regulation    29
32013R1293  Regulation    33
32014R0215  Regulation     9
32014R0517  Regulation    27
32014R0662  Regulation     2
32014R0666  Regulation     8
32014R0749  Regulation    45
32014R1232  Regulation     3
32015R0531  Regulation    17
32015R0757  Regulation    26
32015R1844  Regulation     9
32018L2002  Directive     13
32018R0208  Regulation     2
32018R0841  Regulation    20
32018R0842  Regulation    17
32019L1161  Directive     12
32020R0852  Regulation    28

Article text length stats (chars):
count      825.000000
mean      1719.783030
std       2540.324809
min         40.000000
25%        484.000000
50%       1106.000000
75%       2103.000000
max      32433.000000

Total cros

In [13]:
with OUTPUT_JSONL.open("w", encoding="utf-8") as fh:
    for record in df.to_dict(orient="records"):
        fh.write(json.dumps(record, ensure_ascii=False) + "\n")

file_size_kb = OUTPUT_JSONL.stat().st_size / 1024
print(f"✓ Exported {len(df):,} article records to: {OUTPUT_JSONL}")
print(f"  File size : {file_size_kb:.1f} KB")

with OUTPUT_JSONL.open(encoding="utf-8") as fh:
    line_count = sum(1 for _ in fh)
print(f"  Line count (verify) : {line_count}")

✓ Exported 825 article records to: eu_climate_articles.jsonl
  File size : 1490.2 KB
  Line count (verify) : 825


In [14]:
print("First 3 records in output JSONL:\n")
with OUTPUT_JSONL.open(encoding="utf-8") as fh:
    for i, line in enumerate(fh):
        if i >= 3:
            break
        record = json.loads(line)
        print(f"── Record {i+1} ──")
        for k, v in record.items():
            display = str(v)[:120] + "…" if len(str(v)) > 120 else v
            print(f"  {k:20s}: {display}")
        print()

First 3 records in output JSONL:

── Record 1 ──
  celex_id            : 32003L0030
  doc_type            : Directive
  article_number      : Article 1
  article_text        : Article 1 This Directive aims at promoting the use of biofuels or other renewable fuels to replace diesel or petrol for …
  cross_references    : []

── Record 2 ──
  celex_id            : 32003L0030
  doc_type            : Directive
  article_number      : Article 2
  article_text        : Article 2 1. For the purpose of this Directive, the following definitions shall apply: (a) "biofuels" means liquid or ga…
  cross_references    : []

── Record 3 ──
  celex_id            : 32003L0030
  doc_type            : Directive
  article_number      : Article 3
  article_text        : Article 3 1. (a) Member States should ensure that a minimum proportion of biofuels and other renewable fuels is placed o…
  cross_references    : []



---
## Pipeline Complete

| Stage | Output |
|-------|--------|
| Batch SPARQL | CELEX IDs + Cellar work URIs |
| Per-doc SPARQL | fmx4 / html manifestation URLs (cached in `manifest_cache.json`) |
| Fetch | Formex zips + HTML cached in `raw_cache/` |
| Parse | Article-level records |
| Export | `eu_climate_articles.jsonl` |

**Downstream next steps:**
- **BM25 index** — ingest `article_text` lines into `rank_bm25` or Elasticsearch.
- **Dense embeddings** — encode with `voyage-law-2` or `text-embedding-3-large`; store in Qdrant / pgvector.
- **Hybrid retrieval** — combine BM25 + cosine scores via Reciprocal Rank Fusion (RRF) as the first-stage retrieval layer.